# Data Prep for Neural Network

In [8]:
import polars as pl

## Train Test Split

Here we perform the split into train, validation and test data, with a **split** of 70% train, 15% validation and 15% test data. The split is performed **random**. 

In [9]:
DATASET = "../data/processed_data/GOLD_HOURLY_DEMAND_COMMUNITY_AREA.parquet"
OUTPUT = "../data/train_test_data/"
TARGET_COL = "trip_count"

SEED = 42
RANDOM = False

In [23]:
if(RANDOM == True):
    # split randomly
    df_split = (
        pl.scan_parquet(DATASET)
        .with_row_index("_row_id")
        .with_columns(
            (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
        )
    )

    train = (
        df_split
        .filter(pl.col("_split_bucket") < 70)
        .drop(["_row_id", "_split_bucket"])
    )

    val = (
        df_split
        .filter(
            (pl.col("_split_bucket") >= 70) &
            (pl.col("_split_bucket") < 85)
        )
        .drop(["_row_id", "_split_bucket"])
    )

    test = (
        df_split
        .filter(pl.col("_split_bucket") >= 85)
        .drop(["_row_id", "_split_bucket"])
    )
else :
    # split according to time
    df_split = pl.scan_parquet(DATASET)

    train = df_split.filter(
        pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
    )

    val = df_split.filter(
        (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)) &
        (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
    )

    test = df_split.filter(
        pl.col("datetime_hour") >= pl.datetime(2026, 1, 1)
    )


total_count = df_split.select(pl.len()).collect().item()
train_count = train.select(pl.len()).collect().item()
val_count = val.select(pl.len()).collect().item()
test_count = test.select(pl.len()).collect().item()

print("Total:", total_count)
print("Train:", train_count, " Share: ", round(train_count / total_count,2))
print("Val:", val_count, " Share: ", round(val_count / total_count,2))
print("Test:", test_count, " Share: ", round(test_count / total_count, 2))

train.sink_parquet(OUTPUT + "train.parquet")
val.sink_parquet(OUTPUT + "val.parquet")
test.sink_parquet(OUTPUT + "test.parquet")

Total: 1574265
Train: 1125278  Share:  0.71
Val: 225456  Share:  0.14
Test: 223531  Share:  0.14


In [18]:
type(train)

polars.lazyframe.frame.LazyFrame

In [4]:
df_split.head(10).collect()

_row_id,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,_split_bucket
u32,datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u64
0,2025-12-07 14:00:00,12,7,14,-0.5,0.866025,-0.781831,0.62349,-0.5,-0.866025,-4.44,68.05,11.0,10.0,0.0,1,0,0,0,0,0,2025-12-07,0,47,0.0,0.0,0.0,1.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",55
1,2025-12-10 00:00:00,12,3,0,-0.5,0.866025,0.974928,-0.222521,0.0,1.0,3.89,82.07,18.0,8.0,0.0001,0,0,0,1,0,0,2025-12-10,0,47,0.0,0.0,0.0,1.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",30
2,2025-12-07 15:00:00,12,7,15,-0.5,0.866025,-0.781831,0.62349,-0.707107,-0.707107,-5.0,66.47,13.0,10.0,0.0,0,0,0,0,1,0,2025-12-07,0,47,0.0,0.0,0.0,1.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",67
3,2025-12-19 09:00:00,12,5,9,-0.5,0.866025,-0.433884,-0.900969,0.707107,-0.707107,-10.56,55.35,15.0,10.0,0.0,0,0,0,0,1,0,2025-12-19,0,47,0.0,0.0,0.0,1.0,1,1793,1793.0,1793,1793,13.33,13.33,13.33,13.33,46.06,46.06,46.06,46.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46.06,46.06,46.06,46.06,"""Cash""",81
4,2025-12-07 04:00:00,12,7,4,-0.5,0.866025,-0.781831,0.62349,0.866025,0.5,-1.11,98.0,5.5,0.5,1.77,0,0,0,0,0,1,2025-12-07,0,47,0.0,0.0,0.0,1.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",82
5,2026-01-09 03:00:00,1,5,3,0.0,1.0,-0.433884,-0.900969,0.707107,0.707107,14.44,71.92,25.0,10.0,0.0,0,0,0,0,1,0,2026-01-09,0,47,0.0,0.0,0.0,1.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",13
6,2026-01-08 07:00:00,1,4,7,0.0,1.0,0.433884,-0.900969,0.965926,-0.258819,7.78,67.89,9.0,10.0,0.0,0,0,1,0,0,0,2026-01-08,0,47,0.0,0.0,0.0,1.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",50
7,2026-01-11 21:00:00,1,7,21,0.0,1.0,-0.781831,0.62349,-0.707107,0.707107,-3.33,81.07,11.0,10.0,0.0,0,0,0,0,1,0,2026-01-11,0,47,0.0,0.0,0.0,1.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",1
8,2025-12-03 12:00:00,12,3,12,-0.5,0.866025,0.974928,-0.222521,1.2246e-16,-1.0,-1.835,85.47,10.5,1.75,0.0002,1,0,0,0,0,0,2025-12-03,0,47,0.0,0.0,0.0,1.0,2,4375,2187.5,1207,3168,56.31,28.155,9.54,46.77,176.93,88.465,30.05,146.88,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,176.93,88.465,30.05,146.88,"""Cash""",29


## Feature Selection